In [ ]:
from pathlib import Path
import polars as pl
from dtale import show
import dtale.global_state as global_state
import polars.selectors as cs

global_state.set_app_settings(dict(max_column_width=300))

In [ ]:

data_dir = Path().absolute() / ".." / "data" / "summaries_user_matches"
df = pl.read_parquet(data_dir / "cm0obsd450000xdhhpc50ggcz.snappy")

d = show(df.to_pandas())
d.open_browser()

In [ ]:
summaries_user_matches = df
config = {
    "similarity_threshold": 0.5,
    "social_likelihood_threshold": 0.5,
    "max_summaries_per_user": 10,
}

# Sort by group keys
summaries_user_matches = summaries_user_matches.with_columns(
    avg_social_likelihood=pl.col("social_likelihoods").apply(lambda x: sum(x) / 2),
).sort(
    by=["other_user_id", "cosine_similarity", "avg_social_likelihood"],
    descending=True,
)

# Mask summaries that are too dissimilar or too low in social likelihood
filtered_summaries_user_matches = summaries_user_matches.with_columns(
    mask1=(
        (pl.col("cosine_similarity") < config["similarity_threshold"])
        | (pl.col("avg_social_likelihood") < config["social_likelihood_threshold"])
    ),
    # Mask summaries that are over the max number of summaries per user
).with_columns(
    mask2=(
        pl.int_range(pl.len()).over(["other_user_id", "mask1"])
        >= config["max_summaries_per_user"]
    )
)

show(filtered_summaries_user_matches.to_pandas())


In [ ]:
filtered_summaries_user_matches.filter(pl.col("mask1").not_() & pl.col("mask2").not_()).group_by(
    "other_user_id"
).agg(pl.count()).sort("count", descending=True).to_dicts()